# Neural importance sampling

Reproduce Figures 3(b), 5 and 6 and the timing comparison in Section 5.

In [ ]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/hnsbi_asimov")
    repo = project / "repository"
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/rafaellopesdesa/hnsbi_asimov.git", str(repo)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")],
        check=True,
    )
else:
    repo = Path.cwd()
    project = repo
sys.path.insert(0, str(repo))
from utils import setup_workspace

setup_workspace(repo, project / "workspace")

## Load $q_{\boldsymbol{\phi}}$ and $r_{s,\boldsymbol{\psi}}$

In [ ]:
import gc
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logsumexp
import torch
from IPython.display import display
from utils import (
    FEATURES,
    predict_with_model,
    evaluate_ratio_packs,
    load_ratio_pack,
    sample_selected_flow,
)
from utils_nf import checkpoint_path, flow_log_prob_x, load_flow, train_flow

FEATURES = list(FEATURES)
N_DIM = len(FEATURES)
SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
NIS_MODEL_DIR = Path("models_flows_asimov_nis_influence_v2")
NIS_PLOT_DIR = Path("plots_asimov_nis_influence_v2")
NIS_CACHE_DIR = Path("saved_asimov_nis_influence_v2")
for directory in [NIS_MODEL_DIR, NIS_PLOT_DIR, NIS_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
RATIO_EVALUATION_BATCH_SIZE = 100000
REFERENCE_FLOW_TYPE = "quadratic_spline"
REFERENCE_SAMPLING_BATCH_SIZE = 65536
ASIMOV_MU_TRUE = 1.0
MU_DESIGN = np.linspace(0.0, 3.0, 13)
MU_SCAN = np.linspace(0.0, 3.0, 61)
PILOT_EVENTS = 2000000
NIS_TARGET_CLIP_QUANTILE = 0.9999
NIS_TARGET_FLOOR_FRACTION = 1e-05
DEFENSIVE_REFERENCE_FRACTION = 0.02
ACCEPTANCE_CALIBRATION_EVENTS = 250000
BENCHMARK_EVENTS = 1000000
STUDY_SAMPLE_SIZES = np.asarray([512, 1024, 2048, 4096, 8192, 16384, 32768], dtype=int)
N_REPETITIONS = 64
SHOWCASE_SAMPLE_SIZE = 2048
NIS_MODEL_CONFIG = {
    "flow_type": "quadratic_spline",
    "n_features": N_DIM,
    "n_coupling_layers": 12,
    "hidden_features": 1024,
    "hidden_layers": 4,
    "spline_num_bins": 24,
    "spline_tail_bound": 6.0,
    "dropout_probability": 0.0,
}
NIS_TRAINING_CONFIG = {
    "batch_size": 4096,
    "n_epochs": 70,
    "learning_rate": 0.0001,
    "lr_scheduler_factor": 0.2,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1e-07,
    "weight_decay": 0.0,
    "validation_fraction": 0.2,
    "patience": 10,
    "gradient_clip": 5.0,
}

In [ ]:
PRESEL_pack = load_ratio_pack(PRESEL_MODEL_DIR, 0)


def evaluate_PRESEL_ratio(x):
    return predict_with_model(x, **PRESEL_pack)


state = np.load(PRESEL_MODEL_DIR / "selection.npz")
PRESEL_RATIO_CUT = float(state["ratio_cut"])
LAM_SIG = float(state["nis_lambda_signal"])
LAM_BKG = float(state["nis_lambda_background"])
reference_flow = load_flow(
    "reference", model_dir=REFERENCE_FLOW_MODEL_DIR, flow_type=REFERENCE_FLOW_TYPE, device=device
)
ratio_models = {
    s: [load_ratio_pack(path, member) for member in range(4)] for s, path in RATIO_MODEL_DIR.items()
}


def evaluate_ratio(sample_name, values, batch_size=RATIO_EVALUATION_BATCH_SIZE):
    return evaluate_ratio_packs(ratio_models[sample_name], values, batch_size)


def sample_preselected_flow(flow, n_events, batch_size=65536):
    return sample_selected_flow(
        flow,
        n_events,
        lambda x: evaluate_PRESEL_ratio(x) >= PRESEL_RATIO_CUT,
        batch_size,
        double_needed=True,
    )


def conditional_log_prob(flow, values, acceptance, batch_size=65536):
    return flow_log_prob_x(flow, values, batch_size).astype(np.float64) - np.log(acceptance)

## Evaluate the pilot amplitude $A(\mathbf{x})$ and train $g_{\boldsymbol{\eta}}$ (Algorithm 3)

In [ ]:
def normalized_ratios(raw_signal, raw_background, weights=None):
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()
    signal_norm = float(np.sum(weights * raw_signal))
    background_norm = float(np.sum(weights * raw_background))
    return (raw_signal / signal_norm, raw_background / background_norm)


def scan_influence_amplitude(raw_signal, raw_background, mu_values, weights=None):
    """Influence amplitude of the self-normalized, ratio-normalized scan."""
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()
    mean_signal = float(np.sum(weights * raw_signal))
    mean_background = float(np.sum(weights * raw_background))
    ratio_signal = raw_signal / mean_signal
    ratio_background = raw_background / mean_background
    h_asimov = ASIMOV_MU_TRUE * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
    amplitude_squared = np.zeros(len(ratio_signal), dtype=np.float64)
    scale = ASIMOV_MU_TRUE * LAM_SIG + LAM_BKG
    for mu in np.asarray(mu_values, dtype=np.float64):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        log_h_ratio = np.log(h_asimov / h_mu)
        f_mu = h_asimov * log_h_ratio
        integral_mu = float(np.sum(weights * f_mu))
        h_ratio = h_asimov / h_mu
        derivative_signal = ASIMOV_MU_TRUE * LAM_SIG * (log_h_ratio + 1.0) - mu * LAM_SIG * h_ratio
        derivative_background = LAM_BKG * (log_h_ratio + 1.0 - h_ratio)
        derivative_mean_signal = float(
            np.sum(weights * derivative_signal * (-ratio_signal / mean_signal))
        )
        derivative_mean_background = float(
            np.sum(weights * derivative_background * (-ratio_background / mean_background))
        )
        influence_mu = (
            f_mu
            - integral_mu
            + derivative_mean_signal * (raw_signal - mean_signal)
            + derivative_mean_background * (raw_background - mean_background)
        )
        amplitude_squared += (influence_mu / scale) ** 2
    return np.sqrt(amplitude_squared / len(mu_values))


torch.manual_seed(SEED + 100)
pilot_values, REFERENCE_PRESEL_ACCEPTANCE = sample_preselected_flow(
    reference_flow, PILOT_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
pilot_raw_signal = evaluate_ratio("signal", pilot_values)
pilot_raw_background = evaluate_ratio("background", pilot_values)
pilot_amplitude = scan_influence_amplitude(pilot_raw_signal, pilot_raw_background, MU_DESIGN)
positive_amplitude = pilot_amplitude[pilot_amplitude > 0.0]
amplitude_floor = NIS_TARGET_FLOOR_FRACTION * float(np.median(positive_amplitude))
amplitude_ceiling = float(np.quantile(pilot_amplitude, NIS_TARGET_CLIP_QUANTILE))
pilot_training_amplitude = np.clip(pilot_amplitude, amplitude_floor, amplitude_ceiling)
nis_checkpoint = checkpoint_path("asimov_importance", NIS_MODEL_DIR, NIS_MODEL_CONFIG["flow_type"])
if nis_checkpoint.exists():
    importance_flow = load_flow(
        "asimov_importance",
        model_dir=NIS_MODEL_DIR,
        flow_type=NIS_MODEL_CONFIG["flow_type"],
        device=device,
    )
else:
    importance_training_df = pd.DataFrame(pilot_values, columns=FEATURES)
    importance_flow = train_flow(
        "asimov_importance",
        importance_training_df,
        features=FEATURES,
        model_dir=NIS_MODEL_DIR,
        model_config=NIS_MODEL_CONFIG,
        training_config=NIS_TRAINING_CONFIG,
        device=device,
        sample_weights=pilot_training_amplitude,
        load_if_available=True,
        seed=SEED + 201,
    )
    del importance_training_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
del (
    pilot_values,
    pilot_raw_signal,
    pilot_raw_background,
    pilot_amplitude,
    pilot_training_amplitude,
    positive_amplitude,
)
gc.collect()
torch.manual_seed(SEED + 300)
acceptance_probe_values, IMPORTANCE_PRESEL_ACCEPTANCE = sample_preselected_flow(
    importance_flow, ACCEPTANCE_CALIBRATION_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
del acceptance_probe_values
gc.collect()

## Figure 5: $\log(g_{\boldsymbol{\eta}}/q_{\boldsymbol{\phi}})$ and $\log A$

In [ ]:
VALIDATION_EVENTS = 200000
torch.manual_seed(SEED + 400)
validation_values, _ = sample_preselected_flow(
    reference_flow, VALIDATION_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
validation_raw_signal = evaluate_ratio("signal", validation_values)
validation_raw_background = evaluate_ratio("background", validation_values)
validation_amplitude = scan_influence_amplitude(
    validation_raw_signal, validation_raw_background, MU_DESIGN
)
validation_amplitude = np.clip(validation_amplitude, amplitude_floor, amplitude_ceiling)
validation_log_q = conditional_log_prob(
    reference_flow, validation_values, REFERENCE_PRESEL_ACCEPTANCE
)
validation_log_g = conditional_log_prob(
    importance_flow, validation_values, IMPORTANCE_PRESEL_ACCEPTANCE
)
log_target = np.log(validation_amplitude)
log_proposal_ratio = validation_log_g - validation_log_q
log_target_centered = log_target - np.mean(log_target)
log_proposal_centered = log_proposal_ratio - np.mean(log_proposal_ratio)
proposal_target_correlation = float(np.corrcoef(log_target_centered, log_proposal_centered)[0, 1])
proposal_target_slope = float(np.polyfit(log_target_centered, log_proposal_centered, deg=1)[0])
proposal_target_rmse = float(np.sqrt(np.mean((log_proposal_centered - log_target_centered) ** 2)))
rng = np.random.default_rng(SEED + 401)
plot_indices = rng.choice(
    len(validation_values), size=min(40000, len(validation_values)), replace=False
)
limit = np.quantile(
    np.abs(
        np.concatenate([log_target_centered[plot_indices], log_proposal_centered[plot_indices]])
    ),
    0.995,
)
fig, ax = plt.subplots(figsize=(6.4, 5.5))
hexbin = ax.hexbin(
    log_target_centered[plot_indices],
    log_proposal_centered[plot_indices],
    gridsize=70,
    bins="log",
    mincnt=1,
    cmap="viridis",
)
ax.plot([-limit, limit], [-limit, limit], color="black", ls="--", lw=1.5)
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_xlabel("Centered $\\log A(\\mathbf{x})$")
ax.set_ylabel(
    "Centered $\\log[g_{\\boldsymbol{\\eta}}(\\mathbf{x})/q_{\\boldsymbol{\\phi}}(\\mathbf{x})]$"
)
ax.set_title("Neural proposal versus variance-optimal target")
ax.text(
    0.04,
    0.96,
    f"$\\rho={proposal_target_correlation:.3f}$"
    + "\n"
    + f"slope$={proposal_target_slope:.3f}$"
    + "\n"
    + f"RMS$={proposal_target_rmse:.3f}$",
    transform=ax.transAxes,
    va="top",
)
fig.colorbar(hexbin, ax=ax, label="log count")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "proposal_target_closure.png", dpi=160)
plt.show()

## Draw from $g_\epsilon$ and calculate $\omega_m$ (Algorithm 3)

In [ ]:
def log_defensive_proposal(values):
    log_q = conditional_log_prob(reference_flow, values, REFERENCE_PRESEL_ACCEPTANCE)
    log_g = conditional_log_prob(importance_flow, values, IMPORTANCE_PRESEL_ACCEPTANCE)
    log_mix = logsumexp(
        np.stack(
            [
                np.log(DEFENSIVE_REFERENCE_FRACTION) + log_q,
                np.log1p(-DEFENSIVE_REFERENCE_FRACTION) + log_g,
            ],
            axis=0,
        ),
        axis=0,
    )
    return (log_mix, log_q)


def sample_defensive_proposal(n_events, seed):
    rng = np.random.default_rng(seed)
    n_reference = int(rng.binomial(int(n_events), DEFENSIVE_REFERENCE_FRACTION))
    n_importance = int(n_events) - n_reference
    torch.manual_seed(seed + 1)
    reference_values, _ = (
        sample_preselected_flow(
            reference_flow, n_reference, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
        )
        if n_reference
        else (np.empty((0, N_DIM), dtype=np.float32), np.nan)
    )
    torch.manual_seed(seed + 2)
    importance_values, _ = (
        sample_preselected_flow(
            importance_flow, n_importance, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
        )
        if n_importance
        else (np.empty((0, N_DIM), dtype=np.float32), np.nan)
    )
    values = np.concatenate([reference_values, importance_values], axis=0)
    values = values[rng.permutation(len(values))]
    log_mix, log_q = log_defensive_proposal(values)
    return (values, log_q - log_mix)


def normalized_importance_weights(log_weights):
    log_weights = np.asarray(log_weights, dtype=np.float64)
    return np.exp(log_weights - logsumexp(log_weights))

## Figure 3(b): $\widehat t_A(\boldsymbol{\theta})$ with $M=10^6$

In [ ]:
def asimov_scan(raw_signal, raw_background, mu_values, log_weights=None):
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if log_weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = normalized_importance_weights(log_weights)
    ratio_signal, ratio_background = normalized_ratios(raw_signal, raw_background, weights)
    h_asimov = ASIMOV_MU_TRUE * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
    scan = []
    for mu in np.asarray(mu_values, dtype=np.float64):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        integral = np.sum(weights * h_asimov * np.log(h_asimov / h_mu))
        statistic = 2.0 * ((mu - ASIMOV_MU_TRUE) * LAM_SIG + integral)
        scan.append(max(0.0, float(statistic)))
    return np.asarray(scan)


def evaluate_hybrid_ratios(values):
    return (evaluate_ratio("signal", values), evaluate_ratio("background", values))


torch.manual_seed(SEED + 600)
benchmark_values, _ = sample_preselected_flow(
    reference_flow, BENCHMARK_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
benchmark_raw_signal, benchmark_raw_background = evaluate_hybrid_ratios(benchmark_values)
del benchmark_values
gc.collect()
BENCHMARK_SCAN = asimov_scan(benchmark_raw_signal, benchmark_raw_background, MU_SCAN)
zero_index = int(np.argmin(np.abs(MU_SCAN)))
BENCHMARK_Q_ZERO = float(BENCHMARK_SCAN[zero_index])
fig, ax = plt.subplots(figsize=(6.5, 4.8))
ax.plot(MU_SCAN, BENCHMARK_SCAN, color="black", lw=2.4)
ax.axvline(ASIMOV_MU_TRUE, color="0.5", ls="--", lw=1.2)
ax.set_xlabel("$\\mu$")
ax.set_ylabel("$t_{\\mu,A}$")
ax.set_title("High-statistics hybrid-model Asimov benchmark")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "asimov_benchmark_scan.png", dpi=160)
plt.show()

## Figure 6: convergence of $q_{0,A}$ and $\widehat t_A(\boldsymbol{\theta})$ with $M$

In [ ]:
maximum_study_size = int(np.max(STUDY_SAMPLE_SIZES))
rows = []
for repetition in range(N_REPETITIONS):
    torch.manual_seed(SEED + 10000 + repetition)
    direct_values, _ = sample_preselected_flow(
        reference_flow, maximum_study_size, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
    )
    direct_raw_signal, direct_raw_background = evaluate_hybrid_ratios(direct_values)
    importance_values, importance_log_weights = sample_defensive_proposal(
        maximum_study_size, SEED + 20000 + repetition
    )
    importance_raw_signal, importance_raw_background = evaluate_hybrid_ratios(importance_values)
    for sample_size in STUDY_SAMPLE_SIZES:
        sample_size = int(sample_size)
        for method, raw_signal, raw_background, log_weights in [
            (
                "Direct reference",
                direct_raw_signal[:sample_size],
                direct_raw_background[:sample_size],
                None,
            ),
            (
                "Neural importance",
                importance_raw_signal[:sample_size],
                importance_raw_background[:sample_size],
                importance_log_weights[:sample_size],
            ),
        ]:
            scan = asimov_scan(raw_signal, raw_background, MU_SCAN, log_weights=log_weights)
            q_zero = float(scan[zero_index])
            scan_rmse = float(np.sqrt(np.mean((scan - BENCHMARK_SCAN) ** 2)))
            rows.append(
                {
                    "method": method,
                    "repetition": repetition,
                    "sample_size": sample_size,
                    "q_zero": q_zero,
                    "q_zero_error": q_zero - BENCHMARK_Q_ZERO,
                    "scan_rmse": scan_rmse,
                }
            )
study_results = pd.DataFrame(rows)


def summarize_group(group):
    return pd.Series(
        {
            "q0_mean": group["q_zero"].mean(),
            "q0_bias": group["q_zero_error"].mean(),
            "q0_std": group["q_zero"].std(ddof=1),
            "q0_rmse": np.sqrt(np.mean(group["q_zero_error"] ** 2)),
            "scan_rmse": np.sqrt(np.mean(group["scan_rmse"] ** 2)),
        }
    )


summary_rows = []
for (method, sample_size), group in study_results.groupby(["method", "sample_size"], sort=False):
    row = summarize_group(group).to_dict()
    row.update({"method": method, "sample_size": int(sample_size)})
    summary_rows.append(row)
study_summary = pd.DataFrame(summary_rows)
direct_variance = (
    study_summary.loc[study_summary["method"] == "Direct reference"].set_index("sample_size")[
        "q0_std"
    ]
    ** 2
)
importance_rows = study_summary[study_summary["method"] == "Neural importance"].copy()
importance_rows["variance_reduction"] = (
    importance_rows["sample_size"].map(direct_variance) / importance_rows["q0_std"] ** 2
)
study_summary = study_summary.merge(
    importance_rows[["sample_size", "variance_reduction"]], on="sample_size", how="left"
)
colors = {"Direct reference": "C3", "Neural importance": "C0"}
markers = {"Direct reference": "o", "Neural importance": "s"}
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
for method, group in study_summary.groupby("method", sort=False):
    group = group.sort_values("sample_size")
    axes[0].plot(
        group["sample_size"],
        group["q0_rmse"],
        marker=markers[method],
        color=colors[method],
        lw=2,
        label=method,
    )
    axes[1].plot(
        group["sample_size"],
        group["scan_rmse"],
        marker=markers[method],
        color=colors[method],
        lw=2,
        label=method,
    )
nis_summary = study_summary[study_summary["method"] == "Neural importance"].sort_values(
    "sample_size"
)
axes[2].plot(
    nis_summary["sample_size"], nis_summary["variance_reduction"], marker="D", color="C2", lw=2
)
axes[2].axhline(1.0, color="black", ls="--", lw=1.2)
for axis in axes[:2]:
    axis.set_xscale("log", base=2)
    axis.set_yscale("log")
    axis.grid(alpha=0.25)
    axis.legend()
axes[2].set_xscale("log", base=2)
axes[2].grid(alpha=0.25)
axes[0].set_xlabel("Number of Asimov points")
axes[0].set_ylabel("RMSE of $q_{0,A}$")
axes[0].set_title("Discovery statistic")
axes[1].set_xlabel("Number of Asimov points")
axes[1].set_ylabel("RMS error over $t_{\\mu,A}$")
axes[1].set_title("Complete likelihood scan")
axes[2].set_xlabel("Number of Asimov points")
axes[2].set_ylabel("$\\mathrm{Var}_{q_{\\boldsymbol{\\phi}}}/\\mathrm{Var}_{\\rm NIS}$")
axes[2].set_title("Approximate event-saving factor")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "nis_asimov_convergence.png", dpi=160)
plt.show()

## Section 5: time $\widehat t_A(\boldsymbol{\theta})$ at matched precision

In [ ]:
from scipy.optimize import minimize_scalar
from time import perf_counter

FIT_TIMING_REPETITIONS = 200
FIT_TIMING_WARMUPS = 5
END_TO_END_TIMING_REPETITIONS = 8
END_TO_END_TIMING_WARMUPS = 1
MINIMIZER_BOUNDS = (float(np.min(MU_SCAN)), float(np.max(MU_SCAN)))
MINIMIZER_XATOL = 1e-07
nis_precision_row = study_summary[
    (study_summary["method"] == "Neural importance")
    & (study_summary["sample_size"] == SHOWCASE_SAMPLE_SIZE)
].iloc[0]
direct_precision_candidates = study_summary[study_summary["method"] == "Direct reference"].copy()
direct_precision_candidates["match_distance"] = np.abs(
    np.log(direct_precision_candidates["q0_std"] / float(nis_precision_row["q0_std"]))
)
direct_precision_row = direct_precision_candidates.sort_values("match_distance").iloc[0]
MATCHED_NIS_SIZE = int(nis_precision_row["sample_size"])
MATCHED_DIRECT_SIZE = int(direct_precision_row["sample_size"])


def prepare_asimov_objective(raw_signal, raw_background, log_weights=None):
    """Return the un-clipped finite-quadrature Asimov objective."""
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if log_weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = normalized_importance_weights(log_weights)
    ratio_signal, ratio_background = normalized_ratios(raw_signal, raw_background, weights)
    h_asimov = ASIMOV_MU_TRUE * LAM_SIG * ratio_signal + LAM_BKG * ratio_background

    def objective(mu):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        integral = np.sum(weights * h_asimov * np.log(h_asimov / h_mu))
        return float(2.0 * ((mu - ASIMOV_MU_TRUE) * LAM_SIG + integral))

    return objective


def minimize_asimov(raw_signal, raw_background, log_weights=None):
    objective = prepare_asimov_objective(raw_signal, raw_background, log_weights=log_weights)
    return minimize_scalar(
        objective, bounds=MINIMIZER_BOUNDS, method="bounded", options={"xatol": MINIMIZER_XATOL}
    )


def synchronize_accelerator():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def time_repeated(call, repetitions, warmups):
    for i in range(warmups):
        call(i)
    durations = []
    for i in range(repetitions):
        synchronize_accelerator()
        start = perf_counter()
        call(i + warmups)
        synchronize_accelerator()
        durations.append(1000 * (perf_counter() - start))
    return np.median(durations)


def construct_and_minimize(method, n_events, seed):
    if method == "Direct reference":
        torch.manual_seed(seed)
        values, _ = sample_preselected_flow(reference_flow, n_events)
        log_weights = None
    else:
        values, log_weights = sample_defensive_proposal(n_events, seed)
    rs, rb = evaluate_hybrid_ratios(values)
    return minimize_asimov(rs, rb, log_weights)


timing_rows = []
for method, size, rs, rb, log_weights, precision in [
    (
        "Direct reference",
        MATCHED_DIRECT_SIZE,
        direct_raw_signal,
        direct_raw_background,
        None,
        direct_precision_row,
    ),
    (
        "Neural importance",
        MATCHED_NIS_SIZE,
        importance_raw_signal,
        importance_raw_background,
        importance_log_weights,
        nis_precision_row,
    ),
]:
    args = (rs[:size], rb[:size], None if log_weights is None else log_weights[:size])
    scan_ms = time_repeated(
        lambda i: asimov_scan(args[0], args[1], MU_SCAN, args[2]),
        FIT_TIMING_REPETITIONS,
        FIT_TIMING_WARMUPS,
    )
    minimum_ms = time_repeated(
        lambda i: minimize_asimov(*args), FIT_TIMING_REPETITIONS, FIT_TIMING_WARMUPS
    )
    full_ms = time_repeated(
        lambda i: construct_and_minimize(method, size, SEED + 40000 + i),
        END_TO_END_TIMING_REPETITIONS,
        END_TO_END_TIMING_WARMUPS,
    )
    timing_rows.append(
        {
            "method": method,
            "M": size,
            "q0_std": precision["q0_std"],
            "scan_rmse": precision["scan_rmse"],
            "scan_ms": scan_ms,
            "minimum_ms": minimum_ms,
            "construction_minimum_ms": full_ms,
        }
    )
timing_results = pd.DataFrame(timing_rows)
for column in ["scan_ms", "minimum_ms", "construction_minimum_ms"]:
    timing_results[column.replace("_ms", "_speedup")] = (
        timing_results.loc[0, column] / timing_results[column]
    )
display(timing_results)